# MDP - YOLO26 Training (Local / Offline)

Trains an [Ultralytics YOLO26](https://docs.ultralytics.com/models/yolo26/) detector on **your own machine** (NVIDIA GPU / CUDA) using the `ultralytics` package. No Google Drive or Colab required.

YOLO26 is trained through the `ultralytics` package (`from ultralytics import YOLO`) - there is no separate training repo to clone. The only network access this workflow needs is a **one-time** download of the pretrained COCO starting checkpoint (`yolo26s.pt`) and the initial `pip install`.

# Step 1: Install & Verify Environment

Run the following once in a terminal (from the repo root or anywhere):

```bash
python -m venv .venv
source .venv/bin/activate      # Windows: .venv\Scripts\activate
pip install --upgrade pip
pip install ultralytics
```

If PyTorch is not already CUDA-enabled, install the CUDA build that matches your driver, e.g. for CUDA 12.1:

```bash
pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
```

Then run the cell below to confirm the GPU is visible.

In [1]:
# %pip install ultralytics   # only if you are NOT using a venv

import torch
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

import ultralytics
ultralytics.checks()

Ultralytics 8.4.140  Python-3.14.7 torch-2.14.0+cu132 CUDA:0 (NVIDIA GeForce RTX 5080, 16303MiB)
Setup complete  (16 CPUs, 61.6 GB RAM, 1373.5/1906.6 GB disk)


# Step 2: Point to Your Dataset

Ultralytics uses the same YOLO dataset format as the old v5/v8 notebooks (`images/`, `labels/`, `data.yaml`), so a Roboflow export for v5/v8 works unchanged. Set the path to your dataset's `data.yaml` below. The `train`/`val`/`test` paths inside the yaml are resolved relative to the yaml file itself.

In [3]:
from pathlib import Path

# Absolute or repo-relative path to your dataset's data.yaml
# DATA_YAML = str(Path(r"C:\Users\ryano\Downloads\ntu\SC2079-GROUP31-ImageRec\master_dataset\data_global.yaml").resolve())   # <- change this to your dataset
DATA_YAML = str(Path(r"C:\Users\ryano\Downloads\ntu\SC2079-GROUP31-ImageRec\whitebg_dataset\data.yaml").resolve())
# Where Ultralytics writes checkpoints (relative to this notebook's working dir)
PROJECT = "runs_7(white)"

print("data.yaml:", DATA_YAML)
print("exists:", Path(DATA_YAML).exists())

data.yaml: C:\Users\ryano\Downloads\ntu\SC2079-GROUP31-ImageRec\whitebg_dataset\data.yaml
exists: True


# Step 3: Load the Pretrained Starting Point

The first run downloads the chosen checkpoint from the Ultralytics GitHub release and keeps it locally, so later runs are fully offline. Sizes: `yolo26n.pt` `yolo26s.pt` `yolo26m.pt` `yolo26l.pt` `yolo26x.pt`.

In [2]:

from ultralytics import YOLO

# Pick the biggest size your GPU allows
model = YOLO(r"yolo26n.pt")
# model = YOLO(r"C:\Users\ryano\Downloads\ntu\SC2079-GROUP31-ImageRec\Notebooks\Model Training Notebooks\runs\detect\runs_5(global)\train\weights\best.pt")

In [6]:
!nvidia-smi

Tue Sep 15 23:03:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 610.62                 KMD Version: 610.62        CUDA UMD Version: 13.3     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5080      WDDM  |   00000000:01:00.0 Off |                  N/A |
|  0%   40C    P8             27W /  360W |       0MiB /  16303MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Step 4: Train

- **epochs**: ~20-100 is plenty for this symbol task.
- **batch**: start with `-1` (auto-picks the largest batch your GPU fits), then set a fixed value such as `128` if you want reproducibility.
- **imgsz**: `416` (as used in the previous notebooks) or `640`.
- **fliplr=0.0** is important - horizontal flips would confuse the Left/Right arrow symbols.
- **resume**: if a run is interrupted, load `runs/train/weights/last.pt` and call `model.train(resume=True)`.
- Checkpoints are written to `<PROJECT>/train/weights/` as `best.pt` (best validation) and `last.pt` (final epoch).

In [5]:
model.train(data=DATA_YAML, epochs=20, imgsz=416, batch=-1,
            fliplr=0.0, project=PROJECT, name="train", device=0,resume=False,close_mosaic=2)

New https://pypi.org/project/ultralytics/8.4.153 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.140  Python-3.14.7 torch-2.14.0+cu132 CUDA:0 (NVIDIA GeForce RTX 5080, 16303MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=2, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\ryano\Downloads\ntu\SC2079-GROUP31-ImageRec\whitebg_dataset\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000002A3F4563070>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,

# Step 5: Validate & Test the Trained Model

In [7]:
# Evaluate the best checkpoint on the validation split
best = YOLO(r"C:\Users\ryano\Downloads\ntu\SC2079-GROUP31-ImageRec\Notebooks\Model Training Notebooks\runs\detect\runs_7(white)\train-2\weights\best.pt")
best.val(data=DATA_YAML)

# Run detection on your test images and save the annotated results
# TEST_SOURCE = str(Path("Week8/test/images").resolve())  # <- point at your test folder
best.predict(source=r"C:\Users\ryano\Downloads\Telegram Desktop\New folder", conf=0.5, save=True)

Ultralytics 8.4.140  Python-3.14.7 torch-2.14.0+cu132 CUDA:0 (NVIDIA GeForce RTX 5080, 16303MiB)
YOLO26n summary (fused): 122 layers, 2,382,831 parameters, 0 gradients, 5.4 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 383.8133.9 MB/s, size: 29.0 KB)
val: Scanning C:\Users\ryano\Downloads\ntu\SC2079-GROUP31-ImageRec\whitebg_dataset\valid\labels.cache... 4021 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4021/4021 1.7Git/s 0.0s
WARNING Box and segment counts should be equal, but got len(segments) = 806, len(boxes) = 4026. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 252/252 21.6it/s 11.7s0.0s
                   all       4021       4026      0.993      0.987      0.994      0.962
              Bullseye         58         59      0.982     

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 depth: None
 keypoints: None
 masks: None
 names: {0: 'Unused_0', 1: 'Unused_1', 2: 'Unused_2', 3: 'Unused_3', 4: 'Unused_4', 5: 'Unused_5', 6: 'Unused_6', 7: 'Unused_7', 8: 'Unused_8', 9: 'Unused_9', 10: 'Bullseye', 11: 'One', 12: 'Two', 13: 'Three', 14: 'Four', 15: 'Five', 16: 'Six', 17: 'Seven', 18: 'Eight', 19: 'Nine', 20: 'A', 21: 'B', 22: 'C', 23: 'D', 24: 'E', 25: 'F', 26: 'G', 27: 'H', 28: 'S', 29: 'T', 30: 'U', 31: 'V', 32: 'W', 33: 'X', 34: 'Y', 35: 'Z', 36: 'Up', 37: 'Down', 38: 'Right', 39: 'Left', 40: 'Stop'}
 obb: None
 orig_img: array([[[124, 140, 122],
         [127, 143, 126],
         [125, 142, 128],
         ...,
         [144, 150, 155],
         [146, 149, 157],
         [153, 156, 164]],
 
        [[124, 140, 123],
         [124, 140, 123],
         [128, 145, 131],
         ...,
         [144, 150, 155],
         [145, 148, 156],
         [150, 153, 161

In [21]:
# Evaluate the best checkpoint on the validation split
best = YOLO(r"C:\Users\ryano\Downloads\ntu\SC2079-GROUP31-ImageRec\Notebooks\Model Training Notebooks\runs\detect\runs_5(global)\train\weights\best.pt")
# best.val(data=DATA_YAML)

# Run detection on your test images and save the annotated results
# TEST_SOURCE = str(Path("Week8/test/images").resolve())  # <- point at your test folder
best.predict(source=r"C:\Users\ryano\Downloads\Telegram Desktop\New folder", conf=0.5, save=True)


image 1/19 C:\Users\ryano\Downloads\Telegram Desktop\New folder\2.jpg: 320x416 1 Two, 13.0ms
image 2/19 C:\Users\ryano\Downloads\Telegram Desktop\New folder\2b.jpg: 320x416 (no detections), 13.1ms
image 3/19 C:\Users\ryano\Downloads\Telegram Desktop\New folder\2b2.jpg: 320x416 (no detections), 13.2ms
image 4/19 C:\Users\ryano\Downloads\Telegram Desktop\New folder\3b.jpg: 320x416 1 B, 13.0ms
image 5/19 C:\Users\ryano\Downloads\Telegram Desktop\New folder\4b.jpg: 320x416 (no detections), 13.4ms
image 6/19 C:\Users\ryano\Downloads\Telegram Desktop\New folder\5.jpg: 320x416 1 Five, 13.0ms
image 7/19 C:\Users\ryano\Downloads\Telegram Desktop\New folder\7b.jpg: 320x416 1 Seven, 12.9ms
image 8/19 C:\Users\ryano\Downloads\Telegram Desktop\New folder\b.jpg: 320x416 1 B, 12.7ms
image 9/19 C:\Users\ryano\Downloads\Telegram Desktop\New folder\bb.jpg: 320x416 (no detections), 13.2ms
image 10/19 C:\Users\ryano\Downloads\Telegram Desktop\New folder\e.jpg: 320x416 1 B, 13.2ms
image 11/19 C:\Users\rya

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 depth: None
 keypoints: None
 masks: None
 names: {0: 'unused', 1: 'Box', 2: 'Bullseye', 3: 'NULL', 4: 'unused', 5: 'unused', 6: 'unused', 7: 'unused', 8: 'unused', 9: 'unused', 10: 'unused', 11: 'One', 12: 'Two', 13: 'Three', 14: 'Four', 15: 'Five', 16: 'Six', 17: 'Seven', 18: 'Eight', 19: 'Nine', 20: 'A', 21: 'B', 22: 'C', 23: 'D', 24: 'E', 25: 'F', 26: 'G', 27: 'H', 28: 'S', 29: 'T', 30: 'U', 31: 'V', 32: 'W', 33: 'X', 34: 'Y', 35: 'Z', 36: 'Up', 37: 'Down', 38: 'Right', 39: 'Left', 40: 'Stop'}
 obb: None
 orig_img: array([[[124, 140, 122],
         [127, 143, 126],
         [125, 142, 128],
         ...,
         [144, 150, 155],
         [146, 149, 157],
         [153, 156, 164]],
 
        [[124, 140, 123],
         [124, 140, 123],
         [128, 145, 131],
         ...,
         [144, 150, 155],
         [145, 148, 156],
         [150, 153, 161]],
 
        [[122, 139,

In [4]:
# Evaluate the best checkpoint on the validation split
best = YOLO(r"C:\Users\ryano\Downloads\ntu\SC2079-GROUP31-ImageRec\Notebooks\Model Training Notebooks\runs\detect\runs_3(small_dataset)\train\weights\best.pt")
# best.val(data=DATA_YAML)

# Run detection on your test images and save the annotated results
# TEST_SOURCE = str(Path("Week8/test/images").resolve())  # <- point at your test folder
best.predict(source=r"C:\Users\ryano\Downloads\Telegram Desktop\New folder", conf=0.5, save=True)


image 1/19 C:\Users\ryano\Downloads\Telegram Desktop\New folder\2.jpg: 320x416 1 two, 6.3ms
image 2/19 C:\Users\ryano\Downloads\Telegram Desktop\New folder\2b.jpg: 320x416 (no detections), 7.0ms
image 3/19 C:\Users\ryano\Downloads\Telegram Desktop\New folder\2b2.jpg: 320x416 1 two, 6.1ms
image 4/19 C:\Users\ryano\Downloads\Telegram Desktop\New folder\3b.jpg: 320x416 1 three, 6.2ms
image 5/19 C:\Users\ryano\Downloads\Telegram Desktop\New folder\4b.jpg: 320x416 1 four, 6.4ms
image 6/19 C:\Users\ryano\Downloads\Telegram Desktop\New folder\5.jpg: 320x416 1 five, 6.6ms
image 7/19 C:\Users\ryano\Downloads\Telegram Desktop\New folder\7b.jpg: 320x416 1 seven, 6.2ms
image 8/19 C:\Users\ryano\Downloads\Telegram Desktop\New folder\b.jpg: 320x416 1 B, 6.1ms
image 9/19 C:\Users\ryano\Downloads\Telegram Desktop\New folder\bb.jpg: 320x416 (no detections), 6.1ms
image 10/19 C:\Users\ryano\Downloads\Telegram Desktop\New folder\e.jpg: 320x416 1 E, 6.6ms
image 11/19 C:\Users\ryano\Downloads\Telegram Des

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 depth: None
 keypoints: None
 masks: None
 names: {0: 'A', 1: 'B', 2: 'Bullseye', 3: 'C', 4: 'D', 5: 'E', 6: 'F', 7: 'G', 8: 'H', 9: 'S', 10: 'T', 11: 'U', 12: 'V', 13: 'W', 14: 'X', 15: 'Y', 16: 'Z', 17: 'circle', 18: 'down', 19: 'eight', 20: 'five', 21: 'four', 22: 'left', 23: 'nine', 24: 'one', 25: 'right', 26: 'seven', 27: 'six', 28: 'three', 29: 'two', 30: 'up'}
 obb: None
 orig_img: array([[[124, 140, 122],
         [127, 143, 126],
         [125, 142, 128],
         ...,
         [144, 150, 155],
         [146, 149, 157],
         [153, 156, 164]],
 
        [[124, 140, 123],
         [124, 140, 123],
         [128, 145, 131],
         ...,
         [144, 150, 155],
         [145, 148, 156],
         [150, 153, 161]],
 
        [[122, 139, 125],
         [121, 138, 124],
         [125, 142, 129],
         ...,
         [145, 152, 155],
         [148, 154, 159],
        

# Step 6: Deploy to the Inference Server

Copies `best.pt` into the inference server directory as the checkpoint that `load_model()` in `model.py` expects. The model.py path is `YOLO26_Week_9.pt` (Task 2 / left-right-bullseye); for Task 1 use `YOLO26_Week_8.pt` and update the path in `model.py`.

In [ ]:
import shutil
from pathlib import Path

# The inference server folder is 2 levels up from this notebook
server_dir = Path.cwd().resolve().parents[1] / "YOLOv5 Inference Server"
dest = server_dir / "YOLO26_Week_9.pt"   # use YOLO26_Week_8.pt for Task 1
shutil.copy(f"{PROJECT}/train/weights/best.pt", dest)
print("Copied best.pt to", dest)

## Notes

- **Offline caveat**: `yolo26s.pt` is fetched once over the network on first run (it is cached next to your script/notebook). Everything after that - training, validation, inference - is fully local.
- Runs are saved under `runs/` next to this notebook. Delete old `runs/train/*` folders or change `name=` to avoid collisions.
- The vendored YOLOv5 files in the inference server folder are unused legacy code; training here only needs the `ultralytics` package.